# Research 41 — ExtraTrees fresh-fold blend check

Evaluate the predeclared 80/20 reference + ExtraTrees VIP × region OOF blend from CV seed 2025. Compare with the seed-42 blend using macro and weighted F1 and fold-slice deltas. These are repeated OOF predictions on the same training set, not an independent test. No fitting or Test access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    sums = probs.sum(axis=1, keepdims=True)
    assert (sums > 0).all(), filename
    return probs / sums


p_base = load_probs("segment-oof.csv")
p_seed42 = load_probs("segment-lightgbm-extra-trees-vip-region-blend-oof.csv")
p_candidate = load_probs("segment-lightgbm-extra-trees-vip-region-seed-2025-oof.csv")
p_seed2025 = 0.8 * p_base + 0.2 * p_candidate
folds = {
    seed: list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)
    )
    for seed in [42, 2025]
}
candidates = {
    "Current reference": p_base,
    "ExtraTrees 80/20 blend, seed 42": p_seed42,
    "ExtraTrees 80/20 blend, seed 2025": p_seed2025,
}
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for seed, split in folds.items():
        for fold, (_, idx) in enumerate(split, start=1):
            row[f"seed{seed}_fold{fold}_macro_f1"] = f1_score(y[idx], pred[idx], average="macro")
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-extra-trees-fresh-fold-screen.csv", index=False)
pd.DataFrame(p_seed2025, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-lightgbm-extra-trees-vip-region-seed-2025-blend-oof.csv",
    index=False,
)
print("Fixed 80/20 fresh-fold blend evaluated; repeated OOF screen only, no Test access.")

,candidate,macro_f1,weighted_f1,seed42_fold1_macro_f1,seed42_fold2_macro_f1,seed42_fold3_macro_f1,seed2025_fold1_macro_f1,seed2025_fold2_macro_f1,seed2025_fold3_macro_f1
0,Current reference,0.750895,0.764683,0.749445,0.749821,0.753390,0.750797,0.749412,0.752462
1,"ExtraTrees 80/20 blend, seed 42",0.751649,0.765154,0.750324,0.751090,0.753508,0.751341,0.750872,0.752724
2,"ExtraTrees 80/20 blend, seed 2025",0.751296,0.764788,0.750142,0.749758,0.753961,0.750961,0.750016,0.752902


Fixed 80/20 fresh-fold blend evaluated; repeated OOF screen only, no Test access.


## Interpretation

This seed-2025 refit checks whether the fixed ExtraTrees contribution generalizes across a second stratified partition. The results still reuse the same 104,000 training labels and repeated OOF model selection; they are not an independent holdout or promotion criterion.